# 02b. Timing-aware placebo и диагностика временного выравнивания

Этот ноутбук готовит данные для плацебо-проверки и диагностики временного выравнивания новостей. Здесь статьи распределяются по окнам публикации, формируются семейства сигналов `legacy`, `clean`, `intraday`, `unknown`, `after_close`, `pre_open` и `non_trading`, а затем собирается итоговая таблица для дальнейшего анализа

По умолчанию пересборка модельных выходов отключена: ноутбук использует уже готовые локальные файлы и не должен заново запускать модели без явного изменения флага `FORCE_REBUILD_MODEL_OUTPUTS`

## Блок 1. Импорт библиотек

In [1]:
import os
import gc
import json
import warnings
from datetime import time
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

warnings.simplefilter("default")
pd.set_option("display.max_columns", 250)
pd.set_option("display.width", 220)
np.random.seed(42)

## Блок 2. Конфигурация проекта

Задаём пути, торговые часы и параметры обработки. Пересборка модельных выходов по умолчанию отключена

In [2]:
START_DATE = pd.Timestamp("2019-01-01")
END_DATE = pd.Timestamp("2023-12-31")

OUTPUT_DIR = "outputs_01"

NEWS_PATH = os.path.join(OUTPUT_DIR, "news_deduped_2019_2023.csv")
RET_PATH = os.path.join(OUTPUT_DIR, "returns_sentiment_enhanced.parquet")
SELECTED_EQUITIES_PATH = os.path.join(OUTPUT_DIR, "selected_equities_only_2019_2023.csv")
SELECTED_VARIANTS_PATH = os.path.join(OUTPUT_DIR, "selected_text_variants.json")

MASTER_OUT_PATH = os.path.join(OUTPUT_DIR, "returns_sentiment_timing_aware.parquet")
DIAG_OUT_PATH = os.path.join(OUTPUT_DIR, "timing_alignment_diagnostics.csv")
SUMMARY_OUT_PATH = os.path.join(OUTPUT_DIR, "timing_alignment_summary.json")

TIMING_DIR = os.path.join(OUTPUT_DIR, "timing_patch")
os.makedirs(TIMING_DIR, exist_ok=True)

CHUNK_SIZE = 100000
MARKET_TZ = "America/New_York"
MARKET_OPEN = time(9, 30)
MARKET_CLOSE = time(16, 0)

MODEL_SPECS = {
    "finbert_tone": {
        "kind": "hf",
        "model_id": "yiyanghkust/finbert-tone",
        "max_length": 192,
    },
}

FORCE_REBUILD_MODEL_OUTPUTS = False

if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

HF_BATCH_SIZE = 24 if DEVICE == "mps" else (16 if DEVICE == "cuda" else 8)

print("Устройство:", DEVICE)
print("Путь к новостям:", NEWS_PATH)
print("Путь к доходностям:", RET_PATH)
print("Итоговый timing-aware файл:", MASTER_OUT_PATH)
print("Каталог timing patch:", TIMING_DIR)
print("Модели:", list(MODEL_SPECS.keys()))

Устройство: mps
Путь к новостям: outputs_01/news_deduped_2019_2023.csv
Путь к доходностям: outputs_01/returns_sentiment_enhanced.parquet
Итоговый timing-aware файл: outputs_01/returns_sentiment_timing_aware.parquet
Каталог timing patch: outputs_01/timing_patch
Модели: ['finbert_tone']


## Блок 3. Проверка входных файлов и загрузка базовых таблиц

Проверяем наличие входных данных, загружаем список компаний и базовую таблицу доходностей

In [3]:
required_paths = [NEWS_PATH, RET_PATH, SELECTED_EQUITIES_PATH]
missing = [p for p in required_paths if not os.path.exists(p)]
if missing:
    raise FileNotFoundError(f"Не найдены входные файлы: {missing}")

companies = pd.read_csv(SELECTED_EQUITIES_PATH, low_memory=False)
companies["ticker"] = companies["ticker"].astype(str).str.strip()
TICKER_SET = set(companies["ticker"].tolist())

ret = pd.read_parquet(RET_PATH)
ret["ticker"] = ret["ticker"].astype(str).str.strip()
ret["date"] = pd.to_datetime(ret["date"], errors="coerce").dt.normalize()
ret = ret.dropna(subset=["ticker", "date"]).copy()
ret = ret[ret["ticker"].isin(TICKER_SET)].copy()
ret = ret.sort_values(["ticker", "date"], kind="mergesort").reset_index(drop=True)

for y_col in ["y1_ex", "y2_ex", "y3_ex", "y5_ex"]:
    if y_col not in ret.columns:
        raise KeyError(f"В файле {RET_PATH} нет столбца '{y_col}'. Сначала запусти основной ноутбук 02.")

trade = (
    ret[["ticker", "date"]]
    .drop_duplicates()
    .rename(columns={"date": "trade_date"})
    .sort_values(["ticker", "trade_date"], kind="mergesort")
    .reset_index(drop=True)
)

with open(SELECTED_VARIANTS_PATH, "r", encoding="utf-8") as f:
    selected_variants = json.load(f).get("selected_variants", {})

DEFAULT_VARIANT = selected_variants.get("nasdaq", "title_article")

print("Выбранный текстовый вариант:", DEFAULT_VARIANT)
print("Число тикеров:", len(TICKER_SET))
print("Строк в таблице доходностей:", f"{len(ret):,}")
print("Строк в торговом календаре:", f"{len(trade):,}")

display(companies.head())
display(ret.head())

Выбранный текстовый вариант: title_article
Число тикеров: 2054
Строк в таблице доходностей: 2,580,236
Строк в торговом календаре: 2,580,236


,ticker,company_name,sector,industry,exchange,quoteType,price_rows,price_min_date,price_max_date,coverage_pct,starts_in_2019,ends_in_2023,full_coverage,news_count,news_min_date,news_max_date
0,TSLA,"Tesla, Inc.",Consumer Cyclical,Auto Manufacturers,NMS,EQUITY,1257,2019-01-02,2023-12-28,100.0,True,True,True,10587,2019-07-01,2023-12-16
1,GOOG,Alphabet Inc.,Communication Services,Internet Content & Information,NMS,EQUITY,1257,2019-01-02,2023-12-28,100.0,True,True,True,9883,2019-01-02,2023-12-16
2,DIS,The Walt Disney Company,Communication Services,Entertainment,NYQ,EQUITY,1257,2019-01-02,2023-12-28,100.0,True,True,True,9654,2019-06-13,2023-12-16
3,BHFAL,"Brighthouse Financial, Inc.",NaN,NaN,NMS,EQUITY,1257,2019-01-02,2023-12-28,100.0,True,True,True,9614,2023-11-30,2023-12-16
4,AAPL,Apple Inc.,Technology,Consumer Electronics,NMS,EQUITY,1257,2019-01-02,2023-12-28,100.0,True,True,True,9338,2020-03-09,2023-12-16


,ticker,date,price,ret_log,volume,price_col_used,mkt_ret_log,excess_ret_log,y1_ex,y2_ex,y3_ex,y5_ex,finbert_news_count,finbert_score_mean,finbert_neg_share,finbert_neu_share,finbert_pos_share,finbert_active_score_mean,finbert_active_neg_share,finbert_active_pos_share,news_count,vader_score_mean,vader_neg_share,vader_pos_share,vader_neu_share,vader_active_score_mean,vader_news_count,textblob_score_mean,textblob_neg_share,textblob_pos_share,textblob_neu_share,textblob_active_score_mean,textblob_news_count,finbert_tone_score_mean,finbert_tone_neg_share,finbert_tone_neu_share,finbert_tone_pos_share,finbert_tone_active_score_mean,finbert_tone_active_neg_share,finbert_tone_active_pos_share,finbert_tone_news_count,finroberta_score_mean,finroberta_neg_share,finroberta_neu_share,finroberta_pos_share,finroberta_active_score_mean,finroberta_active_neg_share,finroberta_active_pos_share,finroberta_news_count,distilroberta_finnews_score_mean,distilroberta_finnews_neg_share,distilroberta_finnews_neu_share,distilroberta_finnews_pos_share,distilroberta_finnews_active_score_mean,distilroberta_finnews_active_neg_share,distilroberta_finnews_active_pos_share,distilroberta_finnews_news_count,deberta_finnews_score_mean,deberta_finnews_neg_share,deberta_finnews_neu_share,deberta_finnews_pos_share,deberta_finnews_active_score_mean,deberta_finnews_active_neg_share,deberta_finnews_active_pos_share,deberta_finnews_news_count,twroberta_score_mean,twroberta_neg_share,twroberta_neu_share,twroberta_pos_share,twroberta_active_score_mean,twroberta_active_neg_share,twroberta_active_pos_share,twroberta_news_count,sector,news_n,has_news_today
0,A,2019-01-02,64.968681,NaN,2113300.0,adj close,NaN,NaN,-0.013383,-0.012302,0.000856,0.022114,NaN,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,NaN,0.0,0.0,0.0,0.0,0.0,NaN,0.00,0.0,0.0,0.0,0.00,NaN,0.0,0.0,0.000000e+00,0.0,0.0,0.0,0.0,NaN,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,0.00000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,NaN,Healthcare,0,0
1,A,2019-01-03,62.575249,-0.037536,5383900.0,adj close,-0.024152,-0.013383,0.001081,0.014239,0.019441,0.041318,2.0,0.570507,0.030933,0.367416,0.60144,0.908009,0.045995,0.954005,2.0,0.0,0.0,0.0,1.0,0.0,2.0,0.05,0.0,0.5,0.5,0.05,2.0,1.0,0.0,5.960464e-08,1.0,1.0,0.0,1.0,2.0,0.997358,0.000689,0.001266,0.998047,0.998621,0.000689,0.999311,2.0,0.49968,0.000146,0.500072,0.499826,0.724141,0.137928,0.862069,2.0,0.998218,0.000317,0.001135,0.998535,0.999365,0.000318,0.999682,2.0,0.41049,0.007906,0.57373,0.418396,0.935005,0.032497,0.967503,2.0,Healthcare,2,1
2,A,2019-01-04,64.741203,0.034028,3123700.0,adj close,0.032947,0.001081,0.013158,0.018360,0.034416,0.046694,NaN,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,NaN,0.0,0.0,0.0,0.0,0.0,NaN,0.00,0.0,0.0,0.0,0.00,NaN,0.0,0.0,0.000000e+00,0.0,0.0,0.0,0.0,NaN,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,0.00000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,NaN,Healthcare,0,0
3,A,2019-01-07,66.115929,0.021012,3235100.0,adj close,0.007854,0.013158,0.005202,0.021258,0.027079,0.030664,NaN,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,NaN,0.0,0.0,0.0,0.0,0.0,NaN,0.00,0.0,0.0,0.0,0.00,NaN,0.0,0.0,0.000000e+00,0.0,0.0,0.0,0.0,NaN,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,0.00000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,NaN,Healthcare,0,0
4,A,2019-01-08,67.085175,0.014553,1578100.0,adj close,0.009351,0.005202,0.016056,0.021877,0.028334,0.026179,NaN,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,NaN,0.

## Блок 4. Вспомогательные функции для текста

Определяем функции для нормализации названий столбцов, очистки строк и сборки текстового варианта новости

In [4]:
NEWS_COLS_RAW = [
    "Date",
    "Article_title",
    "Stock_symbol",
    "Url",
    "Publisher",
    "Author",
    "Article",
    "Lsa_summary",
    "Luhn_summary",
    "Textrank_summary",
    "Lexrank_summary",
]

def print_step(title: str):
    line = "=" * 100
    print("\n" + line)
    print(title)
    print(line)

def _clean_str(s: pd.Series) -> pd.Series:
    return s.fillna("").astype(str).str.strip()

def _normalize_news_cols(df: pd.DataFrame) -> pd.DataFrame:
    ren = {}
    if "Date" in df.columns and "date" not in df.columns:
        ren["Date"] = "date"
    if "Stock_symbol" in df.columns and "ticker" not in df.columns:
        ren["Stock_symbol"] = "ticker"
    if "Url" in df.columns and "url" not in df.columns:
        ren["Url"] = "url"
    if "Article_title" in df.columns and "title" not in df.columns:
        ren["Article_title"] = "title"
    if "Article" in df.columns and "article" not in df.columns:
        ren["Article"] = "article"
    if "Lsa_summary" in df.columns and "lsa_summary" not in df.columns:
        ren["Lsa_summary"] = "lsa_summary"
    if "Luhn_summary" in df.columns and "luhn_summary" not in df.columns:
        ren["Luhn_summary"] = "luhn_summary"
    if "Textrank_summary" in df.columns and "textrank_summary" not in df.columns:
        ren["Textrank_summary"] = "textrank_summary"
    if "Lexrank_summary" in df.columns and "lexrank_summary" not in df.columns:
        ren["Lexrank_summary"] = "lexrank_summary"
    if "Publisher" in df.columns and "publisher" not in df.columns:
        ren["Publisher"] = "publisher"
    if "Author" in df.columns and "author" not in df.columns:
        ren["Author"] = "author"
    return df.rename(columns=ren) if ren else df

def _ensure_cols(df: pd.DataFrame, cols: List[str]) -> pd.DataFrame:
    out = df.copy()
    for c in cols:
        if c not in out.columns:
            out[c] = np.nan
    return out

def build_text_variant(df: pd.DataFrame, variant: str) -> pd.Series:
    title = _clean_str(df["title"]) if "title" in df.columns else pd.Series([""] * len(df), index=df.index)
    article = _clean_str(df["article"]) if "article" in df.columns else pd.Series([""] * len(df), index=df.index)

    lsa = _clean_str(df["lsa_summary"]) if "lsa_summary" in df.columns else pd.Series([""] * len(df), index=df.index)
    luhn = _clean_str(df["luhn_summary"]) if "luhn_summary" in df.columns else pd.Series([""] * len(df), index=df.index)
    textrank = _clean_str(df["textrank_summary"]) if "textrank_summary" in df.columns else pd.Series([""] * len(df), index=df.index)
    lexrank = _clean_str(df["lexrank_summary"]) if "lexrank_summary" in df.columns else pd.Series([""] * len(df), index=df.index)

    if variant == "title_article":
        combined = (title + ". " + article).str.replace(r"\s+", " ", regex=True).str.strip()
        out = pd.Series([""] * len(df), index=df.index, dtype="string")
        both = (title != "") & (article != "")
        only_title = (title != "") & (article == "")
        only_article = (title == "") & (article != "")
        out.loc[both] = combined.loc[both]
        out.loc[only_title] = title.loc[only_title]
        out.loc[only_article] = article.loc[only_article]
        return out.astype(str).str.replace(r"\s+", " ", regex=True).str.strip()

    if variant == "lsa_summary":
        return lsa.where(lsa != "", "")
    if variant == "luhn_summary":
        return luhn.where(luhn != "", "")
    if variant == "textrank_summary":
        return textrank.where(textrank != "", "")
    if variant == "lexrank_summary":
        return lexrank.where(lexrank != "", "")

    raise ValueError(f"Неизвестный текстовый вариант: {variant}")

## Блок 5. Подготовка новостного чанка

Читаем очередную часть новостей, собираем текст, сохраняем исходный timestamp и отделяем записи с неизвестным временем

In [5]:
def prep_news_chunk(raw: pd.DataFrame, variant: str = DEFAULT_VARIANT) -> pd.DataFrame:
    chunk = _normalize_news_cols(raw)
    required_cols = [
        "date",
        "ticker",
        "url",
        "title",
        "article",
        "lsa_summary",
        "luhn_summary",
        "textrank_summary",
        "lexrank_summary",
        "publisher",
        "author",
    ]
    chunk = _ensure_cols(chunk, required_cols)

    chunk["ticker"] = _clean_str(chunk["ticker"])
    chunk["url"] = _clean_str(chunk["url"])
    chunk = chunk[(chunk["ticker"] != "") & (chunk["url"] != "")].copy()
    if chunk.empty:
        return chunk.iloc[0:0].copy()

    chunk["ts_utc"] = pd.to_datetime(chunk["date"], errors="coerce", utc=True)
    chunk["published_utc"] = chunk["ts_utc"].dt.tz_convert("UTC").dt.tz_localize(None)

    chunk = chunk.dropna(subset=["published_utc"]).copy()
    if chunk.empty:
        return chunk.iloc[0:0].copy()

    chunk["source_day_utc"] = chunk["published_utc"].dt.normalize()
    chunk = chunk[(chunk["source_day_utc"] >= START_DATE) & (chunk["source_day_utc"] <= END_DATE)].copy()
    if chunk.empty:
        return chunk.iloc[0:0].copy()

    chunk = chunk[chunk["ticker"].isin(TICKER_SET)].copy()
    if chunk.empty:
        return chunk.iloc[0:0].copy()

    chunk = chunk.reset_index(drop=True)

    chunk["time_is_zero_source"] = (
        chunk["ts_utc"].dt.hour.fillna(-1).astype(int).eq(0)
        & chunk["ts_utc"].dt.minute.fillna(-1).astype(int).eq(0)
        & chunk["ts_utc"].dt.second.fillna(-1).astype(int).eq(0)
    )

    chunk["published_et"] = pd.NaT
    known_mask = ~chunk["time_is_zero_source"]
    if known_mask.any():
        chunk.loc[known_mask, "published_et"] = (
            chunk.loc[known_mask, "ts_utc"].dt.tz_convert(MARKET_TZ).dt.tz_localize(None)
        )

    chunk["local_day_et"] = pd.NaT
    if known_mask.any():
        chunk.loc[known_mask, "local_day_et"] = (
            pd.to_datetime(chunk.loc[known_mask, "published_et"], errors="coerce").dt.normalize()
        )

    text = build_text_variant(chunk, variant)
    nonempty = text.fillna("").astype(str).str.strip().ne("")
    chunk = chunk.loc[nonempty].copy()
    if chunk.empty:
        return chunk.iloc[0:0].copy()

    chunk["text"] = text.loc[nonempty].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
    chunk = chunk.drop_duplicates(subset=["ticker", "url", "published_utc"], keep="first").copy()

    keep_cols = [
        "ticker",
        "url",
        "text",
        "published_utc",
        "source_day_utc",
        "published_et",
        "local_day_et",
        "time_is_zero_source",
        "title",
        "article",
        "publisher",
        "author",
    ]
    return chunk[keep_cols].reset_index(drop=True)

## Блок 6. Привязка статей к торговому календарю

Определяем, к какой торговой сессии относится статья, и помечаем её как `legacy`, `clean`, `intraday` или `unknown`

In [6]:
trade_sorted = trade.sort_values(["trade_date", "ticker"], kind="mergesort").reset_index(drop=True)

def add_trade_alignment_features(df_in: pd.DataFrame, trade_df: pd.DataFrame) -> pd.DataFrame:
    out = df_in.copy()
    out["row_id"] = np.arange(len(out), dtype=np.int64)

    legacy_base = (
        out[["row_id", "ticker", "source_day_utc"]]
        .dropna(subset=["source_day_utc"])
        .sort_values(["source_day_utc", "ticker"], kind="mergesort")
    )

    trade_legacy = (
        trade_df[["ticker", "trade_date"]]
        .rename(columns={"trade_date": "legacy_signal_date"})
        .dropna(subset=["legacy_signal_date"])
        .sort_values(["legacy_signal_date", "ticker"], kind="mergesort")
    )

    legacy_map = pd.merge_asof(
        legacy_base,
        trade_legacy,
        by="ticker",
        left_on="source_day_utc",
        right_on="legacy_signal_date",
        direction="forward",
        allow_exact_matches=True,
        tolerance=pd.Timedelta("7D"),
    )[["row_id", "legacy_signal_date"]]

    out = out.merge(legacy_map, on="row_id", how="left")

    known = out[~out["time_is_zero_source"] & out["local_day_et"].notna()].copy()
    if known.empty:
        out["timing_bucket"] = np.where(out["time_is_zero_source"], "unknown_time", "unclassified")
        out["close_anchor_date"] = pd.NaT
        out["reaction_trade_date"] = pd.NaT
        out["is_known_time"] = (~out["time_is_zero_source"]).astype("int8")
        out["is_clean_bucket"] = 0
        return out

    base = (
        known[["row_id", "ticker", "local_day_et"]]
        .dropna(subset=["local_day_et"])
        .sort_values(["local_day_et", "ticker"], kind="mergesort")
    )

    trade_next = (
        trade_df[["ticker", "trade_date"]]
        .rename(columns={"trade_date": "next_trade_date"})
        .dropna(subset=["next_trade_date"])
        .sort_values(["next_trade_date", "ticker"], kind="mergesort")
    )

    next_same_or_next = pd.merge_asof(
        base,
        trade_next,
        by="ticker",
        left_on="local_day_et",
        right_on="next_trade_date",
        direction="forward",
        allow_exact_matches=True,
        tolerance=pd.Timedelta("7D"),
    )[["row_id", "next_trade_date"]]

    trade_prev = (
        trade_df[["ticker", "trade_date"]]
        .rename(columns={"trade_date": "prev_trade_date"})
        .dropna(subset=["prev_trade_date"])
        .sort_values(["prev_trade_date", "ticker"], kind="mergesort")
    )

    prev_strict = pd.merge_asof(
        base,
        trade_prev,
        by="ticker",
        left_on="local_day_et",
        right_on="prev_trade_date",
        direction="backward",
        allow_exact_matches=False,
        tolerance=pd.Timedelta("14D"),
    )[["row_id", "prev_trade_date"]]

    trade_next_strict = (
        trade_df[["ticker", "trade_date"]]
        .rename(columns={"trade_date": "next_trade_after_day"})
        .dropna(subset=["next_trade_after_day"])
        .sort_values(["next_trade_after_day", "ticker"], kind="mergesort")
    )

    next_strict = pd.merge_asof(
        base,
        trade_next_strict,
        by="ticker",
        left_on="local_day_et",
        right_on="next_trade_after_day",
        direction="forward",
        allow_exact_matches=False,
        tolerance=pd.Timedelta("14D"),
    )[["row_id", "next_trade_after_day"]]

    out = out.merge(next_same_or_next, on="row_id", how="left")
    out = out.merge(prev_strict, on="row_id", how="left")
    out = out.merge(next_strict, on="row_id", how="left")

    out["timing_bucket"] = "unknown_time"
    known_mask = ~out["time_is_zero_source"] & out["published_et"].notna() & out["local_day_et"].notna()
    same_day_trade = known_mask & out["local_day_et"].eq(out["next_trade_date"])

    local_minutes = pd.Series(np.nan, index=out.index, dtype=float)
    if known_mask.any():
        ts_local = pd.to_datetime(out.loc[known_mask, "published_et"])
        local_minutes.loc[known_mask] = (
            ts_local.dt.hour * 60 + ts_local.dt.minute + ts_local.dt.second / 60.0
        )

    open_min = MARKET_OPEN.hour * 60 + MARKET_OPEN.minute
    close_min = MARKET_CLOSE.hour * 60 + MARKET_CLOSE.minute

    pre_open = same_day_trade & (local_minutes < open_min)
    intraday = same_day_trade & (local_minutes >= open_min) & (local_minutes < close_min)
    after_close = same_day_trade & (local_minutes >= close_min)
    non_trading = known_mask & (~same_day_trade)

    out.loc[pre_open, "timing_bucket"] = "pre_open"
    out.loc[intraday, "timing_bucket"] = "intraday"
    out.loc[after_close, "timing_bucket"] = "after_close"
    out.loc[non_trading, "timing_bucket"] = "non_trading"

    out["close_anchor_date"] = pd.NaT
    out.loc[after_close, "close_anchor_date"] = out.loc[after_close, "local_day_et"]
    out.loc[pre_open | intraday | non_trading, "close_anchor_date"] = out.loc[
        pre_open | intraday | non_trading, "prev_trade_date"
    ]

    out["reaction_trade_date"] = pd.NaT
    out.loc[after_close, "reaction_trade_date"] = out.loc[after_close, "next_trade_after_day"]
    out.loc[pre_open | intraday, "reaction_trade_date"] = out.loc[pre_open | intraday, "local_day_et"]
    out.loc[non_trading, "reaction_trade_date"] = out.loc[non_trading, "next_trade_date"]

    out["is_known_time"] = (~out["time_is_zero_source"]).astype("int8")
    out["is_clean_bucket"] = out["timing_bucket"].isin(["after_close", "pre_open", "non_trading"]).astype("int8")

    return out

## Блок 7. Вспомогательные функции для модели

Готовим загрузку модели, определяем индексы классов тональности и считаем вероятности по батчам

In [7]:
def _cleanup_torch_device():
    try:
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        elif torch.backends.mps.is_available():
            torch.mps.empty_cache()
    except Exception:
        pass

def _load_hf_model(model_id: str):
    tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)
    model = AutoModelForSequenceClassification.from_pretrained(model_id).to(DEVICE)
    model.eval()

    raw_id2lab = getattr(model.config, "id2label", {}) or {}
    id2lab = {int(k): str(v).lower() for k, v in raw_id2lab.items()} if isinstance(raw_id2lab, dict) else {}

    raw_lab2id = getattr(model.config, "label2id", {}) or {}
    lab2id = {str(k).lower(): int(v) for k, v in raw_lab2id.items()} if isinstance(raw_lab2id, dict) else {}
    return tokenizer, model, id2lab, lab2id

def _resolve_sentiment_slots(id2lab: dict, lab2id: dict):
    def pick_from_id2lab(keys):
        for idx, label in id2lab.items():
            label = str(label).lower()
            for key in keys:
                if key in label:
                    return int(idx)
        return None

    def pick_from_lab2id(keys):
        for key in keys:
            for label, idx in lab2id.items():
                if key in str(label).lower():
                    return int(idx)
        return None

    neg = pick_from_id2lab(["negative", "neg", "bearish"])
    pos = pick_from_id2lab(["positive", "pos", "bullish"])
    neu = pick_from_id2lab(["neutral", "neu", "none"])

    if neg is None:
        neg = pick_from_lab2id(["negative", "neg", "bearish"])
    if pos is None:
        pos = pick_from_lab2id(["positive", "pos", "bullish"])
    if neu is None:
        neu = pick_from_lab2id(["neutral", "neu", "none"])

    return neg, neu, pos

def _hf_predict(texts: List[str], tok, mdl, neg_i, neu_i, pos_i, batch_size: int, max_length: int):
    n = len(texts)
    pneg = np.zeros(n, dtype=np.float32)
    pneu = np.zeros(n, dtype=np.float32)
    ppos = np.zeros(n, dtype=np.float32)

    with torch.inference_mode():
        for start in range(0, n, batch_size):
            batch = texts[start:start + batch_size]
            enc = tok(
                batch,
                padding=True,
                truncation=True,
                max_length=int(max_length),
                return_tensors="pt",
            )
            enc = {k: v.to(DEVICE) for k, v in enc.items()}
            logits = mdl(**enc).logits
            probs = torch.softmax(logits, dim=-1).detach().cpu().numpy().astype(np.float32)

            if neg_i is None or pos_i is None:
                if probs.shape[1] == 2:
                    pneg[start:start + len(batch)] = probs[:, 0]
                    ppos[start:start + len(batch)] = probs[:, 1]
                    pneu[start:start + len(batch)] = 0.0
                else:
                    raise ValueError("Не удалось определить индексы классов тональности для модели.")
            else:
                pneg[start:start + len(batch)] = probs[:, int(neg_i)]
                ppos[start:start + len(batch)] = probs[:, int(pos_i)]
                pneu[start:start + len(batch)] = probs[:, int(neu_i)] if neu_i is not None else 0.0

    score = (ppos - pneg).astype(np.float32)
    return score, pneg, pneu, ppos

## Блок 8. Агрегация статей в дневные признаки

Преобразуем статьи в дневные агрегаты по тикеру и дате и считаем распределение наблюдений по временным окнам

In [8]:
def article_metrics_frame(
    scored_df: pd.DataFrame,
    date_col: str,
    score_col: str = "score",
    pneg_col: str = "pneg",
    pneu_col: str = "pneu",
    ppos_col: str = "ppos",
) -> pd.DataFrame:
    cols = ["ticker", date_col, score_col, pneg_col, pneu_col, ppos_col]
    tmp = scored_df[cols].copy()
    tmp = tmp.rename(columns={date_col: "date"})
    tmp = tmp.dropna(subset=["ticker", "date"]).copy()
    if tmp.empty:
        return tmp

    tmp["n"] = 1
    tmp["score_sum"] = pd.to_numeric(tmp[score_col], errors="coerce").astype(float)

    pneg = pd.to_numeric(tmp[pneg_col], errors="coerce").astype(float)
    pneu = pd.to_numeric(tmp[pneu_col], errors="coerce").astype(float)
    ppos = pd.to_numeric(tmp[ppos_col], errors="coerce").astype(float)

    mass = (pneg + ppos).astype(float)
    tmp["neg_sum"] = pneg
    tmp["neu_sum"] = pneu
    tmp["pos_sum"] = ppos
    tmp["active_score_sum"] = tmp["score_sum"] / (mass + 1e-9)
    tmp["active_neg_sum"] = pneg / (mass + 1e-9)
    tmp["active_pos_sum"] = ppos / (mass + 1e-9)

    grouped = (
        tmp.groupby(["ticker", "date"], sort=False)[
            ["n", "score_sum", "neg_sum", "neu_sum", "pos_sum", "active_score_sum", "active_neg_sum", "active_pos_sum"]
        ]
        .sum()
        .reset_index()
    )
    return grouped

def reduce_parts(parts: List[pd.DataFrame]) -> pd.DataFrame:
    if not parts:
        return pd.DataFrame(columns=["ticker", "date"])
    df = pd.concat(parts, ignore_index=True)
    df["ticker"] = df["ticker"].astype(str).str.strip()
    df["date"] = pd.to_datetime(df["date"], errors="coerce").dt.normalize()
    df = df.dropna(subset=["ticker", "date"]).copy()
    num_cols = [c for c in df.columns if c not in ["ticker", "date"]]
    df[num_cols] = df[num_cols].apply(pd.to_numeric, errors="coerce")
    df = df.groupby(["ticker", "date"], sort=False)[num_cols].sum().reset_index()
    return df.sort_values(["ticker", "date"], kind="mergesort").reset_index(drop=True)

def finalize_family_frame(raw_agg: pd.DataFrame, prefix: str) -> pd.DataFrame:
    if raw_agg.empty:
        return pd.DataFrame(columns=["ticker", "date", f"{prefix}_news_count"])

    grouped = raw_agg.copy()
    n = pd.to_numeric(grouped["n"], errors="coerce").astype(float)

    grouped[f"{prefix}_news_count"] = n.round().fillna(0).astype("int32")
    grouped[f"{prefix}_score_mean"] = np.where(n > 0, grouped["score_sum"] / n, np.nan).astype("float32")
    grouped[f"{prefix}_neg_share"] = np.where(n > 0, grouped["neg_sum"] / n, np.nan).astype("float32")
    grouped[f"{prefix}_neu_share"] = np.where(n > 0, grouped["neu_sum"] / n, np.nan).astype("float32")
    grouped[f"{prefix}_pos_share"] = np.where(n > 0, grouped["pos_sum"] / n, np.nan).astype("float32")
    grouped[f"{prefix}_active_score_mean"] = np.where(n > 0, grouped["active_score_sum"] / n, np.nan).astype("float32")
    grouped[f"{prefix}_active_neg_share"] = np.where(n > 0, grouped["active_neg_sum"] / n, np.nan).astype("float32")
    grouped[f"{prefix}_active_pos_share"] = np.where(n > 0, grouped["active_pos_sum"] / n, np.nan).astype("float32")

    keep = ["ticker", "date"] + [c for c in grouped.columns if c.startswith(prefix + "_")]
    return grouped[keep].sort_values(["ticker", "date"], kind="mergesort").reset_index(drop=True)

def timing_bucket_counts(df_in: pd.DataFrame) -> pd.DataFrame:
    if df_in.empty or "timing_bucket" not in df_in.columns:
        return pd.DataFrame(columns=["timing_bucket", "n_articles"])

    out = (
        df_in["timing_bucket"]
        .fillna("missing")
        .value_counts(dropna=False)
        .rename_axis("timing_bucket")
        .reset_index(name="n_articles")
    )
    return out

## Блок 9. Сборка timing-aware дневных панелей

Для выбранной модели читаем новости чанками, считаем тональность, раскладываем статьи по семействам сигналов и сохраняем дневные таблицы

In [9]:
def build_timing_daily_for_hf_model(
    model_tag: str,
    model_id: str,
    max_length: int,
    variant: str = DEFAULT_VARIANT,
    chunk_size: int = CHUNK_SIZE,
    batch_size: int = HF_BATCH_SIZE,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    final_out_path = os.path.join(TIMING_DIR, f"daily_timing_{model_tag}_2019_2023.parquet")
    diag_out_path = os.path.join(TIMING_DIR, f"diag_timing_{model_tag}_2019_2023.csv")

    if os.path.exists(final_out_path) and (not FORCE_REBUILD_MODEL_OUTPUTS):
        daily = pd.read_parquet(final_out_path)
        diag = pd.read_csv(diag_out_path) if os.path.exists(diag_out_path) else pd.DataFrame()
        print(f"[{model_tag}] Загружен готовый timing-aware файл: {final_out_path}")
        return daily, diag

    print_step(f"Сборка timing-aware дневной панели для {model_tag}")

    tok, mdl, id2lab, lab2id = _load_hf_model(model_id)
    neg_i, neu_i, pos_i = _resolve_sentiment_slots(id2lab, lab2id)

    family_parts: Dict[str, List[pd.DataFrame]] = {
        "legacy": [],
        "clean": [],
        "intraday": [],
        "unknown": [],
        "after_close": [],
        "pre_open": [],
        "non_trading": [],
    }

    diag_rows = []
    reader = pd.read_csv(NEWS_PATH, chunksize=chunk_size, low_memory=False)

    for chunk_no, raw in enumerate(tqdm(reader, desc=f"{model_tag}: обработка чанков")):
        chunk = prep_news_chunk(raw, variant=variant)
        if chunk.empty:
            continue

        chunk = add_trade_alignment_features(chunk, trade_sorted)
        if chunk.empty:
            continue

        texts = chunk["text"].astype(str).tolist()
        score, pneg, pneu, ppos = _hf_predict(
            texts=texts,
            tok=tok,
            mdl=mdl,
            neg_i=neg_i,
            neu_i=neu_i,
            pos_i=pos_i,
            batch_size=batch_size,
            max_length=max_length,
        )

        chunk["score"] = score
        chunk["pneg"] = pneg
        chunk["pneu"] = pneu
        chunk["ppos"] = ppos

        family_masks = {
            "legacy": chunk["legacy_signal_date"].notna(),
            "clean": chunk["is_clean_bucket"].eq(1) & chunk["close_anchor_date"].notna(),
            "intraday": chunk["timing_bucket"].eq("intraday") & chunk["close_anchor_date"].notna(),
            "unknown": chunk["timing_bucket"].eq("unknown_time") & chunk["legacy_signal_date"].notna(),
            "after_close": chunk["timing_bucket"].eq("after_close") & chunk["close_anchor_date"].notna(),
            "pre_open": chunk["timing_bucket"].eq("pre_open") & chunk["close_anchor_date"].notna(),
            "non_trading": chunk["timing_bucket"].eq("non_trading") & chunk["close_anchor_date"].notna(),
        }
        family_date_col = {
            "legacy": "legacy_signal_date",
            "clean": "close_anchor_date",
            "intraday": "close_anchor_date",
            "unknown": "legacy_signal_date",
            "after_close": "close_anchor_date",
            "pre_open": "close_anchor_date",
            "non_trading": "close_anchor_date",
        }

        for family, mask in family_masks.items():
            if bool(mask.any()):
                part = article_metrics_frame(chunk.loc[mask].copy(), date_col=family_date_col[family])
                if not part.empty:
                    family_parts[family].append(part)

        bucket_table = timing_bucket_counts(chunk)
        bucket_table["chunk_no"] = int(chunk_no)
        bucket_table["n_articles_chunk"] = int(len(chunk))
        diag_rows.append(bucket_table)

    del mdl
    _cleanup_torch_device()
    gc.collect()

    merged_daily = None
    family_final = {}

    for family, parts in family_parts.items():
        raw_agg = reduce_parts(parts)
        prefix = f"{model_tag}_{family}"
        family_final[family] = finalize_family_frame(raw_agg, prefix=prefix)
        if merged_daily is None:
            merged_daily = family_final[family]
        else:
            cols = [c for c in family_final[family].columns if c not in ["ticker", "date"]]
            merged_daily = merged_daily.merge(
                family_final[family][["ticker", "date"] + cols],
                on=["ticker", "date"],
                how="outer",
                validate="one_to_one",
            )

    if merged_daily is None:
        merged_daily = pd.DataFrame(columns=["ticker", "date"])

    merged_daily = merged_daily.sort_values(["ticker", "date"], kind="mergesort").reset_index(drop=True)

    diag = pd.concat(diag_rows, ignore_index=True) if diag_rows else pd.DataFrame()
    if not diag.empty:
        diag = (
            diag.groupby("timing_bucket", as_index=False)[["n_articles", "n_articles_chunk"]]
            .sum()
            .sort_values("n_articles", ascending=False)
        )
        diag["share_articles"] = diag["n_articles"] / max(int(diag["n_articles"].sum()), 1)

    merged_daily.to_parquet(final_out_path, index=False)
    diag.to_csv(diag_out_path, index=False)

    print(f"[{model_tag}] Сохранён файл: {final_out_path}")
    print(f"[{model_tag}] Число строк: {len(merged_daily):,}")

    return merged_daily, diag

## Блок 10. Запуск расчёта по выбранной модели

Запускаем построение timing-aware панелей и сохраняем диагностические таблицы по временным окнам

In [10]:
timing_model_outputs = {}
timing_model_diags = {}

for model_tag, cfg in MODEL_SPECS.items():
    if cfg["kind"] != "hf":
        raise NotImplementedError("В этом ноутбуке реализован только путь с моделями Hugging Face.")

    daily, diag = build_timing_daily_for_hf_model(
        model_tag=model_tag,
        model_id=cfg["model_id"],
        max_length=int(cfg["max_length"]),
        variant=DEFAULT_VARIANT,
        chunk_size=CHUNK_SIZE,
        batch_size=HF_BATCH_SIZE,
    )
    timing_model_outputs[model_tag] = daily
    timing_model_diags[model_tag] = diag

for tag, diag in timing_model_diags.items():
    print("\nДиагностика распределения по временным окнам для", tag)
    display(diag)

[finbert_tone] Загружен готовый timing-aware файл: outputs_01/timing_patch/daily_timing_finbert_tone_2019_2023.parquet

Диагностика распределения по временным окнам для finbert_tone


,timing_bucket,n_articles,n_articles_chunk,share_articles
0,unknown_time,1208133,1230315,0.981970
1,pre_open,11109,150345,0.009029
2,non_trading,7010,1032138,0.005698
3,after_close,3927,993072,0.003192
4,intraday,136,129053,0.000111


## Блок 11. Сборка итоговой мастер-таблицы

Объединяем базовую панель доходностей с timing-aware признаками и сохраняем финальный датасет для плацебо-анализа

In [11]:
print_step("Сборка итоговой timing-aware мастер-таблицы")

master = ret.copy()

if "sector" in companies.columns:
    master = master.merge(companies[["ticker", "sector"]], on="ticker", how="left")

for model_tag, daily in timing_model_outputs.items():
    cols = [c for c in daily.columns if c not in ["ticker", "date"]]
    master = master.merge(
        daily[["ticker", "date"] + cols],
        on=["ticker", "date"],
        how="left",
        validate="one_to_one",
    )

main_model = list(MODEL_SPECS.keys())[0]
family_names = ["legacy", "clean", "intraday", "unknown", "after_close", "pre_open", "non_trading"]

for family in family_names:
    count_col = f"{main_model}_{family}_news_count"
    if count_col in master.columns:
        master[count_col] = pd.to_numeric(master[count_col], errors="coerce").fillna(0).astype("int32")
        master[f"has_{family}_news"] = (master[count_col] > 0).astype("int8")

sent_suffixes = [
    "_score_mean",
    "_active_score_mean",
    "_neg_share",
    "_neu_share",
    "_pos_share",
    "_active_neg_share",
    "_active_pos_share",
]

for model_tag in MODEL_SPECS:
    for family in family_names:
        prefix = f"{model_tag}_{family}"
        count_col = f"{prefix}_news_count"
        if count_col not in master.columns:
            continue
        family_sent_cols = [f"{prefix}{suffix}" for suffix in sent_suffixes if f"{prefix}{suffix}" in master.columns]
        no_news_mask = master[count_col].fillna(0).eq(0)
        if family_sent_cols:
            master.loc[no_news_mask, family_sent_cols] = master.loc[no_news_mask, family_sent_cols].fillna(0.0)

master = master.replace([np.inf, -np.inf], np.nan)
master = master.sort_values(["ticker", "date"], kind="mergesort").reset_index(drop=True)

dup_count = int(master.duplicated(["ticker", "date"]).sum())
print("Число дубликатов по (ticker, date):", dup_count)
if dup_count:
    raise ValueError("В итоговой timing-aware таблице есть дубликаты по паре (ticker, date).")

master.to_parquet(MASTER_OUT_PATH, index=False)
print("Сохранён итоговый timing-aware файл:", MASTER_OUT_PATH)
print("Число строк:", f"{len(master):,}")
print("Число столбцов:", master.shape[1])

display(master.head())


Сборка итоговой timing-aware мастер-таблицы
Число дубликатов по (ticker, date): 0
Сохранён итоговый timing-aware файл: outputs_01/returns_sentiment_timing_aware.parquet
Число строк: 2,580,236
Число столбцов: 140


,ticker,date,price,ret_log,volume,price_col_used,mkt_ret_log,excess_ret_log,y1_ex,y2_ex,y3_ex,y5_ex,finbert_news_count,finbert_score_mean,finbert_neg_share,finbert_neu_share,finbert_pos_share,finbert_active_score_mean,finbert_active_neg_share,finbert_active_pos_share,news_count,vader_score_mean,vader_neg_share,vader_pos_share,vader_neu_share,vader_active_score_mean,vader_news_count,textblob_score_mean,textblob_neg_share,textblob_pos_share,textblob_neu_share,textblob_active_score_mean,textblob_news_count,finbert_tone_score_mean,finbert_tone_neg_share,finbert_tone_neu_share,finbert_tone_pos_share,finbert_tone_active_score_mean,finbert_tone_active_neg_share,finbert_tone_active_pos_share,finbert_tone_news_count,finroberta_score_mean,finroberta_neg_share,finroberta_neu_share,finroberta_pos_share,finroberta_active_score_mean,finroberta_active_neg_share,finroberta_active_pos_share,finroberta_news_count,distilroberta_finnews_score_mean,distilroberta_finnews_neg_share,distilroberta_finnews_neu_share,distilroberta_finnews_pos_share,distilroberta_finnews_active_score_mean,distilroberta_finnews_active_neg_share,distilroberta_finnews_active_pos_share,distilroberta_finnews_news_count,deberta_finnews_score_mean,deberta_finnews_neg_share,deberta_finnews_neu_share,deberta_finnews_pos_share,deberta_finnews_active_score_mean,deberta_finnews_active_neg_share,deberta_finnews_active_pos_share,deberta_finnews_news_count,twroberta_score_mean,twroberta_neg_share,twroberta_neu_share,twroberta_pos_share,twroberta_active_score_mean,twroberta_active_neg_share,twroberta_active_pos_share,twroberta_news_count,sector_x,news_n,has_news_today,sector_y,finbert_tone_legacy_news_count,finbert_tone_legacy_score_mean,finbert_tone_legacy_neg_share,finbert_tone_legacy_neu_share,finbert_tone_legacy_pos_share,finbert_tone_legacy_active_score_mean,finbert_tone_legacy_active_neg_share,finbert_tone_legacy_active_pos_share,finbert_tone_clean_news_count,finbert_tone_clean_score_mean,finbert_tone_clean_neg_share,finbert_tone_clean_neu_share,finbert_tone_clean_pos_share,finbert_tone_clean_active_score_mean,finbert_tone_clean_active_neg_share,finbert_tone_clean_active_pos_share,finbert_tone_intraday_news_count,finbert_tone_intraday_score_mean,finbert_tone_intraday_neg_share,finbert_tone_intraday_neu_share,finbert_tone_intraday_pos_share,finbert_tone_intraday_active_score_mean,finbert_tone_intraday_active_neg_share,finbert_tone_intraday_active_pos_share,finbert_tone_unknown_news_count,finbert_tone_unknown_score_mean,finbert_tone_unknown_neg_share,finbert_tone_unknown_neu_share,finbert_tone_unknown_pos_share,finbert_tone_unknown_active_score_mean,finbert_tone_unknown_active_neg_share,finbert_tone_unknown_active_pos_share,finbert_tone_after_close_news_count,finbert_tone_after_close_score_mean,finbert_tone_after_close_neg_share,finbert_tone_after_close_neu_share,finbert_tone_after_close_pos_share,finbert_tone_after_close_active_score_mean,finbert_tone_after_close_active_neg_share,finbert_tone_after_close_active_pos_share,finbert_tone_pre_open_news_count,finbert_tone_pre_open_score_mean,finbert_tone_pre_open_neg_share,finbert_tone_pre_open_neu_share,finbert_tone_pre_open_pos_share,finbert_tone_pre_open_active_score_mean,finbert_tone_pre_open_active_neg_share,finbert_tone_pre_open_active_pos_share,finbert_tone_non_trading_news_count,finbert_tone_non_trading_score_mean,finbert_tone_non_trading_neg_share,finbert_tone_non_trading_neu_share,finbert_tone_non_trading_pos_share,finbert_tone_non_trading_active_score_mean,finbert_tone_non_trading_active_neg_share,finbert_tone_non_trading_active_pos_share,has_legacy_news,has_clean_news,has_intraday_news,has_unknown_news,has_after_close_news,has_pre_open_news,has_non_trading_news
0,A,2019-01-02,64.968681,NaN,2113300.0,adj close,NaN,NaN,-0.013383,-0.012302,0.000856,0.022114,NaN,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,NaN,0.0,0.0,0.0,0.0,0.0,NaN,0.00,0.0,0.0,0.0,0.00,NaN,0.0,0.0,0.000000e+00,0.0,0.0,0.0,0.0,NaN,0.000000,0.00

## Блок 12. Диагностика для плацебо

Сохраняем сводные показатели

In [12]:
summary_rows = []

summary_rows.append(
    {
        "metric": "n_trade_panel_rows",
        "value": int(len(master)),
    }
)

for family in family_names:
    flag_col = f"has_{family}_news"
    if flag_col in master.columns:
        summary_rows.append(
            {
                "metric": f"share_{family}_days_in_master",
                "value": float(pd.to_numeric(master[flag_col], errors="coerce").fillna(0).mean()),
            }
        )

for tag, diag in timing_model_diags.items():
    if diag.empty:
        continue
    for _, row in diag.iterrows():
        summary_rows.append(
            {
                "metric": f"{tag}_share_articles_{row['timing_bucket']}",
                "value": float(row["share_articles"]),
            }
        )
        summary_rows.append(
            {
                "metric": f"{tag}_n_articles_{row['timing_bucket']}",
                "value": int(row["n_articles"]),
            }
        )

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(DIAG_OUT_PATH, index=False)

summary_json = {str(row["metric"]): row["value"] for _, row in summary_df.iterrows()}
with open(SUMMARY_OUT_PATH, "w", encoding="utf-8") as f:
    json.dump(summary_json, f, ensure_ascii=False, indent=2)

print("Сохранён файл:", DIAG_OUT_PATH)
print("Сохранён файл:", SUMMARY_OUT_PATH)

display(summary_df)

Сохранён файл: outputs_01/timing_alignment_diagnostics.csv
Сохранён файл: outputs_01/timing_alignment_summary.json


,metric,value
0,n_trade_panel_rows,2.580236e+06
1,share_legacy_days_in_master,2.071845e-01
2,share_clean_days_in_master,4.434866e-03
3,share_intraday_days_in_master,4.960787e-05
4,share_unknown_days_in_master,2.043693e-01
5,share_after_close_days_in_master,8.867406e-04
6,share_pre_open_days_in_master,3.582230e-03
7,share_non_trading_days_in_master,4.015137e-04
8,finbert_tone_share_articles_unknown_time,9.819705e-01
9,finbert_tone_n_articles_unknown_time,1.208133e+06
